In [ ]:
pip install --upgrade openai

In [ ]:
# =========================================
# OFFLINE HEALTHCARE ASSISTANT (TF-IDF)
# =========================================

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================================
# 1. Medical Knowledge Base
# =========================================
medical_docs = [
    "Diabetes symptoms include frequent urination, fatigue, and increased thirst.",
    "Hypertension is high blood pressure and may require lifestyle changes or medication.",
    "Paracetamol is used to treat fever but overdose can damage the liver.",
    "Asthma causes breathing difficulty, chest tightness, and wheezing.",
    "Ibuprofen is a pain reliever but excessive use can cause stomach problems.",
    "Migraine causes severe headache, nausea, and sensitivity to light.",
    "COVID-19 symptoms include fever, cough, and loss of taste or smell."
]

# =========================================
# 2. TF-IDF Vectorization
# =========================================
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(medical_docs)

# =========================================
# 3. Retrieve Most Relevant Docs
# =========================================
def retrieve(query, top_k=2):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, doc_vectors)

    top_indices = similarity.argsort()[0][-top_k:][::-1]
    results = [medical_docs[i] for i in top_indices]

    return results, similarity[0][top_indices]

# =========================================
# 4. Generate Answer (Smarter Logic)
# =========================================
def generate_answer(context, scores):
    best_score = max(scores)

    if best_score < 0.2:
        return "Insufficient medical data. Please consult a doctor."

    return "Possible condition based on symptoms:\n- " + context[0]

# =========================================
# 5. Safety Layer
# =========================================
def safety_check(response):
    unsafe_words = ["overdose", "high dosage", "guaranteed cure"]

    for word in unsafe_words:
        if word in response.lower():
            return "⚠️ Unsafe advice detected. Please consult a doctor."

    return response

# =========================================
# 6. Confidence Score
# =========================================
def confidence_score(scores):
    return round(float(max(scores)), 2)

# =========================================
# 7. Main Function
# =========================================
def ask(query):
    print("\n🔍 Query:", query)

    context, scores = retrieve(query)
    answer = generate_answer(context, scores)
    safe_answer = safety_check(answer)
    conf = confidence_score(scores)

    print("\n💡 Answer:", safe_answer)
    print(f"📊 Confidence: {conf*100}%")
    print("📚 Top Matches:")
    for c in context:
        print("-", c)
    print("⚠️ Not medical advice\n")

# =========================================
# 8. Run Loop
# =========================================
if __name__ == "__main__":
    while True:
        user_input = input("Enter symptoms (or exit): ")
        if user_input.lower() == "exit":
            break
        ask(user_input)

Enter symptoms (or exit): fever, cough,headache

🔍 Query: fever, cough,headache

💡 Answer: Possible condition based on symptoms:
- COVID-19 symptoms include fever, cough, and loss of taste or smell.
📊 Confidence: 32.0%
📚 Top Matches:
- COVID-19 symptoms include fever, cough, and loss of taste or smell.
- Migraine causes severe headache, nausea, and sensitivity to light.
⚠️ Not medical advice

